In [ ]:
from datetime import datetime, timedelta
import os

# 获取SLS日志的查询结果
import time
import pandas as pd
from sls_client import get_sls_data_by_query
from odps_client import logging


view_sql = """
type:view AND (pageName:/home OR pageName:/goods OR pageName:/search/goods OR pageName:/goods/category) AND bid:"sku:"
| SELECT
  uid,type,page_name,variant_id,sku_id,event_date,pd_name,
  avg(sale_price) avg_sale_price,
  min(sale_price) min_sale_price,
  max(sale_price) max_sale_price,
  avg(idx) avg_idx,
  min(idx) min_idx,
  max(idx) max_idx,
  count(distinct event_time) sku_view_cnt
FROM  (
    SELECT
      uid,
      sid,
      cid,
      type,
      pageName AS page_name,
      COALESCE(abTestVersion, 'none') AS variant_id,
      DATE_FORMAT(__time__, '%Y%m%d %H:%i:%s') AS event_time,
      DATE_FORMAT(__time__, '%Y%m%d') AS event_date,
      cast(REPLACE(REGEXP_EXTRACT(sku_info, 'salePrice:([\d\.]+)'),'salePrice:','') as double) AS sale_price,
      REPLACE(REGEXP_EXTRACT(sku_info, 'name:([^,]+)'),'name:','') AS pd_name,
      cast(REPLACE(REGEXP_EXTRACT(sku_info, 'idx:(\d+)'), 'idx:', '') as bigint) AS idx,
      REPLACE(REGEXP_EXTRACT(sku_info, 'sku:(\w+)'), 'sku:', '') AS sku_id
    FROM      log,
      UNNEST(SPLIT(bid, ';')) AS view_skus(sku_info)
    LIMIT
      10000000
  )
group by
  uid,type,page_name,variant_id,sku_id,event_date,pd_name
"""

add_cart_query = """
ap:/shopping/cart/upsert/insert | select pageName page_name,json_extract(json_extract_scalar(ai, '$.data'),'$.quantity') quantity
,json_extract_scalar(json_extract_scalar(ai, '$.data'),'$.sku') sku,json_extract_scalar(ai, '$.qh["xm-rqid"]') rqid
,coalesce(json_extract_scalar(json_extract_scalar(ai, '$.qh["xm-ab-exp"]'),'$[0].variantId'),'none')variant_id
,json_extract_scalar(json_extract_scalar(ai, '$.rt'), '$.msg') return_msg
,cid,uid,sid,date_format(__time__,'%Y%m%d %H:%i:%s')event_time,date_format(__time__,'%Y%m%d') event_date
from log order by event_time limit 1000000"""


def get_user_data_from_sls_by_query(
    from_time: datetime, to_time: datetime, query: str = add_cart_query
) -> pd.DataFrame:
    data_from_sls_df = None

    data_from_sls_df = get_sls_data_by_query(
        from_time=from_time,
        to_time=to_time,
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
    )
    if data_from_sls_df is None:
        logging.error(f"没有获取到用户的数据:{from_time}~{to_time}")
        return

    return data_from_sls_df


def get_user_data_local_storage_path(ds: str, data_type: str = "add_cart") -> str:
    return f"./data/user_{data_type}_all_df_{ds}.csv"


# 如果CSV文件没有数据，则从get_user_click_data获取数据
def load_user_data_from_local_disk(
    ds_to_run: str, data_type: str = "add_cart"
) -> pd.DataFrame:
    # 尝试从CSV文件读取数据
    csv_file_path = get_user_data_local_storage_path(ds_to_run, data_type=data_type)
    if os.path.exists(csv_file_path):
        df = pd.read_csv(csv_file_path)
        if not df.empty:
            print(f"从CSV文件加载了{len(df)}条记录")
            return df

    print(f"未找到日期:{ds_to_run}的{data_type}数据")
    return pd.DataFrame()


def fetch_data_from_disk_or_sls(
    ds: str,
    date_to_fetch: datetime,
    data_type: str = "add_cart",
    query_to_fetch: str = add_cart_query,
) -> pd.DataFrame:
    ds_df = load_user_data_from_local_disk(ds, data_type=data_type)
    if ds_df.empty:
        # 如果本地缓存没有，则重新跑一遍
        hours_interval = 8
        for hour_index in range(0, 24, hours_interval):  # 8小时一批次

            from_hour = date_to_fetch + timedelta(hours=hour_index)
            to_hour = from_hour + timedelta(hours=hours_interval)  # 8小时一个批次

            df = get_user_data_from_sls_by_query(
                from_time=from_hour, to_time=to_hour, query=query_to_fetch
            )
            ds_df = pd.concat([ds_df, df], ignore_index=True)

        # 如果全部获取完了所有页面的数据，则写入本地缓存
        if not ds_df.empty:
            local_path = get_user_data_local_storage_path(ds=ds, data_type=data_type)
            ds_df.to_csv(local_path, index=False)
            print(
                f"已将 {len(ds_df)} 条记录保存到 {local_path}，总记录数：{len(ds_df)}，日期：{ds}"
            )
    else:
        print(
            f"从本地缓存中获取到了DS:{ds}的数据,数据条数:{len(ds_df)}, pages:{list(ds_df['page_name'].unique())}"
        )

    return ds_df

In [ ]:
start_date = datetime(2024, 9, 27).date()
end_date = datetime(2024, 10, 31).date()
N = (end_date - start_date).days
print(f"start_date:{start_date}, end_date:{end_date}, N:{N}")

user_add_cart_all_df = pd.DataFrame()
user_sku_view_all_df = pd.DataFrame()

for i in range(N):
    from_time = datetime.combine(start_date + timedelta(i), datetime.min.time())
    to_time = from_time + timedelta(hours=24)
    ds = from_time.strftime("%Y%m%d")
    print(f"正在处理 {from_time} 到 {to_time} 的数据，日期：{ds}")

    ds_add_cart_df = fetch_data_from_disk_or_sls(
        ds=ds,
        date_to_fetch=from_time,
        data_type="add_cart",
        query_to_fetch=add_cart_query,
    )
    ds_sku_view_df = fetch_data_from_disk_or_sls(
        ds=ds, date_to_fetch=from_time, data_type="sku_view", query_to_fetch=view_sql
    )

    if not ds_add_cart_df.empty:
        user_add_cart_all_df = pd.concat(
            [user_add_cart_all_df, ds_add_cart_df], ignore_index=True
        )
        del ds_add_cart_df
    if not ds_sku_view_df.empty:
        user_sku_view_all_df = pd.concat(
            [user_sku_view_all_df, ds_sku_view_df], ignore_index=True
        )
        del ds_sku_view_df

In [ ]:
from odps_client import get_odps_sql_result_as_df

category_df=get_odps_sql_result_as_df("""select sku_id,category1 from summerfarm_tech.dim_sku_df 
                                      where ds=max_pt('summerfarm_tech.dim_sku_df');""")

In [ ]:
import pandasql
from datetime import datetime, timedelta
from odps_client import write_pandas_df_into_odps

start_date = datetime(2024, 9, 27)
end_date = datetime(2024, 10, 30)
all_add_cart_summary_df = pd.DataFrame()

while start_date <= end_date:
    ds = start_date.strftime("%Y%m%d")
    print(f"正在处理日期: {ds}")

    df = fetch_data_from_disk_or_sls(
        ds=ds, date_to_fetch=None, data_type="add_cart", query_to_fetch=None
    )

    if not df.empty:
        add_cart_summary_df = pandasql.sqldf(
            """
select cast(uid as bigint)uid,page_name,variant_id,event_date,category1,
sum(quantity) added_quantity,count(distinct a.sku) as added_sku_cnt,count(1) added_times
from df a inner join category_df c on a.sku=c.sku_id
where uid is not null
group by page_name,variant_id,event_date,category1,uid
        """
        )

        add_cart_summary_df["event_date"] = add_cart_summary_df["event_date"].astype(str)

        all_add_cart_summary_df = pd.concat(
            [all_add_cart_summary_df, add_cart_summary_df], ignore_index=True
        )
        del add_cart_summary_df
        del df

    start_date += timedelta(days=1)

print("数据处理完成")
all_add_cart_summary_df.head(20)

partition_spec = f"ds={all_add_cart_summary_df['event_date'].max()}"
write_pandas_df_into_odps(
    df=all_add_cart_summary_df,
    table_name="temp_ab_user_add_cart_summary_df",
    partition_spec=partition_spec,
    overwrite=True,
)

In [ ]:
import pandasql
from datetime import datetime, timedelta
from odps_client import write_pandas_df_into_odps

start_date = datetime(2024, 9, 27)
end_date = datetime(2024, 10, 30)
all_view_summary_df = pd.DataFrame()

while start_date <= end_date:
    ds = start_date.strftime("%Y%m%d")
    print(f"正在处理日期: {ds}")

    df = fetch_data_from_disk_or_sls(
        ds=ds, date_to_fetch=None, data_type="sku_view", query_to_fetch=None
    )

    if not df.empty:
        view_summary_df = pandasql.sqldf(
            """
            select cast(uid as bigint)uid,page_name,variant_id,event_date,category1,
                   sum(sku_view_cnt) sku_view_cnt,count(distinct a.sku_id) as viewed_sku
            from df a inner join category_df c on a.sku_id=c.sku_id
            where uid is not null
            group by page_name,variant_id,event_date,category1,uid
        """
        )

        view_summary_df["event_date"] = view_summary_df["event_date"].astype(str)

        all_view_summary_df = pd.concat(
            [all_view_summary_df, view_summary_df], ignore_index=True
        )
        del view_summary_df
        del df

    start_date += timedelta(days=1)

print("数据处理完成")
all_view_summary_df.head(20)

partition_spec = f"ds={all_view_summary_df['event_date'].max()}"
write_pandas_df_into_odps(
    df=all_view_summary_df,
    table_name="temp_ab_user_sku_view_df",
    partition_spec=partition_spec,
    overwrite=True,
)

In [3]:
from odps_client import get_odps_sql_result_as_df

analytics_sql = """
select a.*,b.added_quantity,b.added_sku_cnt,b.added_times
from summerfarm_ds.temp_ab_user_sku_view_df a
left join summerfarm_ds.temp_ab_user_add_cart_summary_df b
on b.ds=max_pt('summerfarm_ds.temp_ab_user_add_cart_summary_df')
and a.uid=b.uid
and a.category1=b.category1
and a.page_name=b.page_name
and a.event_date=b.event_date
and a.variant_id=b.variant_id
where a.ds=max_pt('summerfarm_ds.temp_ab_user_sku_view_df')
"""

analytics_df = get_odps_sql_result_as_df(sql=analytics_sql)
analytics_df["added_quantity"].fillna(0, inplace=True)
analytics_df["added_quantity"] = analytics_df["added_quantity"].astype(int)

analytics_df["added_sku_cnt"].fillna(0, inplace=True)
analytics_df["added_sku_cnt"] = analytics_df["added_sku_cnt"].astype(int)

analytics_df["added_times"].fillna(0, inplace=True)
analytics_df["added_times"] = analytics_df["added_times"].astype(int)
analytics_df.head(10)

## 以下是点击数据

In [ ]:
user_add_cart_all_df["uid"] = user_add_cart_all_df["uid"].astype(int)
user_add_cart_all_df["event_date"] = user_add_cart_all_df["event_date"].astype(str)
user_add_cart_all_df["quantity"] = user_add_cart_all_df["quantity"].astype(int)
user_add_cart_all_df.drop(
    columns=["__time__", "__source__"], errors="ignore", inplace=True
)

user_add_cart_all_with_category_df = pd.merge(
    left=user_add_cart_all_df,
    right=category_df,
    how="left",
    left_on="sku",
    right_on="sku_id",
    suffixes=["", "_b"],
)

from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps
from datetime import datetime, timedelta


ds_of_today = datetime.now().strftime("%Y%m%d")
partition_spec = f"ds={ds_of_today}"

write_pandas_df_into_odps(
    df=user_add_cart_all_with_category_df,
    table_name="temp_ab_user_add_cart_all_df",
    partition_spec=partition_spec,
    overwrite=True,
)

user_add_cart_all_with_category_df.tail(1)

In [ ]:
import pandas as pd
import glob
import os

# 获取所有匹配的CSV文件路径
csv_files = glob.glob('./data/user_click_all_new_df_2024*.csv')

# 读取所有CSV文件并合并为一个DataFrame
df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    df_list.append(df)
    
user_click_all_new_df = pd.concat(df_list, ignore_index=True)

user_click_all_new_df.head(1)


In [2]:
# 获取所有匹配的CSV文件路径
csv_files = glob.glob('./data/user_add_cart_all_df_2024*.csv')

# 读取所有CSV文件并合并为一个DataFrame
df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    df_list.append(df)
    
user_add_cart_all_df = pd.concat(df_list, ignore_index=True)


In [4]:
import pandasql 

user_add_cart_all_df.tail(2)
user_add_cart_all_df['ds']=user_add_cart_all_df['ds'].astype(str)
user_add_cart_all_df['uid']=user_add_cart_all_df['uid'].astype(int)

sql="""
select ds,uid,pageName as page_name,sku,count(*) add_times,sum(quantity) added_quantity
from user_add_cart_all_df
where return_msg = '请求成功'
group by ds,uid,pageName,sku
"""

user_add_cart_all_df=pandasql.sqldf(sql)

user_add_cart_all_df['ds']=user_add_cart_all_df['ds'].astype(str)
user_add_cart_all_df['uid']=user_add_cart_all_df['uid'].astype(int)

In [ ]:
import re
pd.set_option('display.max_colwidth', None)

# Function to extract `sku` and `pid` from `bid` column if `sku` and `pid` are NaN
def extract_sku_pid(row):
    if pd.isna(row['sku']):
        sku_match = re.search(r'sku:([\da-zA-Z]+)', row['bid'])
        row['sku'] = sku_match.group(1) if sku_match else row['sku']
    if pd.isna(row['pid']):
        pid_match = re.search(r'pid:([\w]+)', row['bid'])
        row['pid'] = pid_match.group(1) if pid_match else row['pid']
    return row

# Apply the function to each row
user_click_all_new_df = user_click_all_new_df.apply(extract_sku_pid, axis=1)

user_click_all_new_df.tail(10)[['bid','sku','pid','ds','uid']]

In [ ]:
user_click_all_new_df.groupby('pid').agg({
    'sku': ['nunique', 'count']
}).reset_index()


In [ ]:
sql1="""
select sku,uid,ds,pageName as page_name,pid
    ,count(*) total_click_cnt 
    ,count(case when pid='唤起购买' then 1 end) as open_card_cnt
    ,count(case when pid='goods' then 1 end) as click_to_goods_cnt
from user_click_all_new_df
group by sku,uid,ds,pageName,pid
"""

user_click_all_new_df_grouped=pandasql.sqldf(sql1)
user_click_all_new_df_grouped['ds']=user_click_all_new_df_grouped['ds'].astype(str)
user_click_all_new_df_grouped['uid']=user_click_all_new_df_grouped['uid'].astype(int)

In [ ]:
user_click_all_new_df_grouped[user_click_all_new_df_grouped['sku'].isna()]['ds'].max()

In [ ]:
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps
from datetime import datetime, timedelta


ds_of_today = datetime.now().strftime("%Y%m%d")
partition_spec = f"ds={ds_of_today}"

user_click_all_new_df_grouped.rename(columns={"ds": "event_date"}, inplace=True)
user_add_cart_all_df.rename(columns={"ds": "event_date"}, inplace=True)

write_pandas_df_into_odps(
    df=user_click_all_new_df_grouped,
    table_name="temp_ab_user_click_all_new_df",
    partition_spec=partition_spec,
    overwrite=True,
)


write_pandas_df_into_odps(
    df=user_add_cart_all_df,
    table_name="temp_ab_user_add_cart_all_df",
    partition_spec=partition_spec,
    overwrite=True,
)

In [ ]:
from odps_client import get_odps_sql_result_as_df

sql_stat="""
WITH user_view_df AS 
(
    SELECT  a.ds
            ,a.cust_id
            ,c.category1
            ,CASE   WHEN b.variant_id IS NOT NULL AND b.variant_id IN ('V3','V4') THEN '实验组用户'
                    ELSE '未进实验组用户'
            END AS is_experienced
            ,page_name
            ,a.sku_id
            ,COUNT(1) AS sku_view_cnt
    FROM    summerfarm_tech.dwd_log_mall_di a
    INNER JOIN   (
                    SELECT  cust_id AS mid
                            ,MIN(variant_id) variant_id
                    FROM    summerfarm_ds.temp_mall_new_home_ab_info_di
                    WHERE   ds between '20240910' and '20241021'
                    GROUP BY cust_id
                ) b
    ON      b.mid = a.cust_id
    INNER JOIN summerfarm_tech.dim_sku_df c
    ON      c.ds = MAX_PT('summerfarm_tech.dim_sku_df')
    AND     c.sku_id = a.sku_id
    WHERE   a.ds between '20240927' and '20241029'
    AND     a.sku_id IS NOT NULL
    AND     a.envent_type = 'view'
    AND     a.cust_id IS NOT NULL
    AND     a.page_name IN ('/goods','/search/goods','/home','/goods/category')
    GROUP BY a.ds
             ,a.cust_id
             ,a.sku_id
             ,is_experienced
             ,a.page_name
             ,c.category1
)
SELECT  a.ds
        ,a.category1
        ,a.page_name
        ,a.is_experienced
        ,SUM(sku_view_cnt) SKU曝光数
        ,count(distinct a.sku_id) 有曝光的SKU个数
        ,SUM(b.click_to_goods_cnt) 商品详情点开数
        ,SUM(c.added_quantity) 加购总件数
        ,count(case when c.added_quantity > 0 then 1 end) 加购API总调用次数
        ,SUM(b.open_card_cnt) 购物车点开次数
        ,COUNT(DISTINCT a.cust_id) 总UV
        ,COUNT(DISTINCT CASE    WHEN c.added_quantity > 0 THEN c.uid END) 加购UV
        ,COUNT(DISTINCT CASE    WHEN c.added_quantity > 0 THEN c.sku END) 加购sku
        ,round(100.00*COUNT(DISTINCT CASE    WHEN c.added_quantity > 0 THEN c.uid END)/count(distinct a.cust_id),2) 加购UV转化率
        ,round(100.00*COUNT(DISTINCT CASE    WHEN b.click_to_goods_cnt > 0 THEN b.uid END)/count(distinct a.cust_id),2) 点击商品详情UV转化率
FROM    user_view_df a
LEFT JOIN summerfarm_ds.temp_ab_user_click_all_new_df b
ON      b.ds = MAX_PT('summerfarm_ds.temp_ab_user_click_all_new_df')
AND     a.ds = b.event_date
AND     a.cust_id = b.uid
AND     a.page_name = b.page_name
AND     a.sku_id = b.sku
LEFT JOIN summerfarm_ds.temp_ab_user_add_cart_all_df c
ON      c.ds = MAX_PT('summerfarm_ds.temp_ab_user_add_cart_all_df')
AND     a.ds = c.event_date
AND     a.cust_id = c.uid
AND     a.page_name = c.page_name
AND     a.sku_id = c.sku
GROUP BY a.ds
         ,a.category1
         ,a.page_name
         ,a.is_experienced
ORDER BY a.ds,a.category1,a.page_name,a.is_experienced
;
"""

page_df=get_odps_sql_result_as_df(sql_stat)

In [ ]:
page_df["人均加购总件数"] = page_df["加购总件数"] / page_df["总uv"]
page_df["加购人群人均加购总件数"] = page_df["加购总件数"] / page_df["加购uv"]
page_df["商品详情ctr"] = 100.00 * page_df["商品详情点开数"] / page_df["sku曝光数"]
page_df["购物车卡片ctr"] = 100.00 * page_df["购物车点开次数"] / page_df["sku曝光数"]
page_df["加购api转化率"] = 100.00 * page_df["加购api总调用次数"] / page_df["sku曝光数"]
page_df["avg件数per已加购sku"] = page_df["加购总件数"] / page_df["加购sku"]
page_df.tail(20)

In [10]:
page_df.to_csv(
    f"./data/商城价格展示优化实验全量之后数据对比_{page_df['ds'].min()}~{page_df['ds'].max()}.csv",
    index=False,
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 设置中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 筛选数据
df = page_df[page_df["ds"] >= "20240927"]

# 将日期转换为datetime类型
df["ds"] = pd.to_datetime(df["ds"])

# 要分析的指标列表
metrics = [
    "人均加购总件数",
    # "已加购用户平均加购SKU件数", 
    # "点击CTR_点开加购卡片",
    # "商品详情ctr",
    # "加购uv转化率",
    "avg件数per已加购sku",
    "加购sku",
]

# 获取所有页面和类目
pages = df["page_name"].unique()
categories = df["category1"].unique()

# 计算需要的子图数量
total_plots = len(metrics) * len(pages) * len(categories)

# 创建子图网格
fig, axes = plt.subplots(total_plots, 1, figsize=(15, 4*total_plots))
plot_idx = 0

# 三层循环：指标、页面、类目
for metric in metrics:
    for page in pages:
        for category in categories:
            # 筛选当前页面和类目的数据
            plot_df = df[
                (df["page_name"] == page) & 
                (df["category1"] == category)
            ]
            
            # 为每个实验组画一条线
            for exp_group in df["is_experienced"].unique():
                exp_df = plot_df[plot_df["is_experienced"] == exp_group]
                axes[plot_idx].plot(
                    exp_df["ds"],
                    exp_df[metric],
                    label=exp_group,
                    marker="o",
                    markersize=4,
                )

            axes[plot_idx].set_title(f"指标:{metric}, 页面:{page}, 类目:{category}", fontsize=10)
            axes[plot_idx].set_xlabel("日期")
            axes[plot_idx].set_ylabel(metric)
            axes[plot_idx].legend(title="实验组", bbox_to_anchor=(1.05, 1), loc="upper left")

            # 设置x轴刻度，每个日期显示一次
            date_ticks = df["ds"].unique()[::1]
            axes[plot_idx].set_xticks(date_ticks)
            axes[plot_idx].set_xticklabels(
                date_ticks.strftime("%Y-%m-%d"), rotation=45, ha="right"
            )

            # 强制y轴从0开始
            axes[plot_idx].set_ylim(bottom=0)

            # 添加网格线
            axes[plot_idx].grid(True, linestyle="--", alpha=0.7)

            plot_idx += 1

plt.tight_layout()
plt.show()

In [ ]:
from odps_client import get_odps_sql_result_as_df

sql="""
select uid,variant_id,event_date,sku,page_name,category1
    ,sum(quantity) added_quantity
    ,count(distinct rqid) rqid_cnt
    ,count(1) rows_cnt
from summerfarm_ds.temp_ab_user_add_cart_all_df
where ds=max_pt('summerfarm_ds.temp_ab_user_add_cart_all_df')
and page_name = '/home' and category1='乳制品'
and event_date <='20241026'
group by uid,event_date,variant_id,sku,page_name,category1
"""

user_add_cart_df=get_odps_sql_result_as_df(sql)
user_add_cart_df.head(5)


In [ ]:
for variant in range(1, 5):
    variant_df = user_add_cart_df[user_add_cart_df["variant_id"] == f"V{variant}"]
    print(variant_df.describe())
    print(
        f"variant:{variant}",
        variant_df["added_quantity"].quantile(
            [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]
        ),
    )